1. Build per-attack annotation CSVs

In [1]:
# Build per-attack annotation CSVs with RELATIVE paths like:
# processed_images\train\img_0.jpg  -->  adv_outputs\<ATTACK>\processed_images\train\img_0.jpg
# Output filenames: Annotations_<ATTACK>.csv

from pathlib import Path
import pandas as pd
import os

CSV_PATH = Path("../../annotations/new_annotations.csv")
ADV_ROOT = Path("../../datasets/adv_outputs")
OUT_DIR  = Path("../../annotations/adv_annotations")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# This is the RELATIVE prefix we want in the new image_id values
REL_ADV_PREFIX = Path("adv_outputs")

def load_csv_safely(path: Path) -> pd.DataFrame:
    """Read CSV robustly (UTF-8 BOM, comma or semicolon)."""
    try:
        df0 = pd.read_csv(path, encoding="utf-8-sig")
        if len(df0.columns) == 1 and ";" in str(df0.columns[0]):
            raise ValueError("semicolon detected")
        return df0
    except Exception:
        return pd.read_csv(path, encoding="utf-8-sig", sep=";")

df = load_csv_safely(CSV_PATH)
assert "image_id" in df.columns, "CSV must contain column 'image_id'"
print("Loaded columns:", list(df.columns))

def rel_tail_from_processed_images(p: str) -> str:
    """
    Return the subpath starting at 'processed_images\\...' (case-insensitive).
    If not found, fallback to 'processed_images\\unknown\\<filename>'.
    """
    s = str(p)
    low = s.lower()
    key = "processed_images"
    i = low.find(key)
    if i >= 0:
        return s[i:].replace("/", "\\")  # normalize to backslashes
    return os.path.join("processed_images", "unknown", os.path.basename(s))

# Discover attack folders under ADV_ROOT
if not ADV_ROOT.exists():
    raise FileNotFoundError(f"ADV_ROOT does not exist: {ADV_ROOT}")
attacks = sorted([p for p in ADV_ROOT.iterdir() if p.is_dir()])
if not attacks:
    raise FileNotFoundError(f"No subfolders found in ADV_ROOT: {ADV_ROOT}")

print("Attacks found:", [a.name for a in attacks])

# --- Force tails to use .png and precompute once ---
def force_png_tail(tail: str) -> str:
    """
    Ensure the filename in the tail ends with .png (replace original extension).
    tail example: 'processed_images\\train\\img_0.jpg' -> 'processed_images\\train\\img_0.png'
    """
    p = Path(str(tail).lstrip("\\/"))
    parent = p.parent
    stem = p.stem
    new_tail = str(parent / (stem + ".png"))
    return new_tail.replace("/", "\\")

# Precompute tails once from original annotation, but force .png extension
raw_tails = df["image_id"].astype(str).apply(rel_tail_from_processed_images)
tails_png = raw_tails.apply(force_png_tail)

for attack_dir in attacks:
    attack_name = attack_dir.name  # e.g., ANN_FGSM_eps002

    # Build RELATIVE image_id: adv_outputs\<ATTACK>\ + tail
    new_ids = [str(Path(REL_ADV_PREFIX) / attack_name / Path(t)) for t in tails_png]

    df_out = df.copy()
    df_out["image_id"] = new_ids
    # Optional: mark split as 'adv' (uncomment if you want)
    # if "split" in df_out.columns:
    #     df_out["split"] = "adv"

    out_csv = OUT_DIR / f"Annotations_{attack_name}.csv"
    df_out.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"Saved: {out_csv}")

print("\nDone. Relative-path annotations are in:", OUT_DIR)


Loaded columns: ['image_id', 'label', 'isNight', 'split']
Attacks found: ['ANN_fgsm', 'ANN_pgd', 'CNN_fgsm', 'CNN_pgd', 'RNN_fgsm', 'RNN_pgd']
Saved: ..\..\annotations\adv_annotations\Annotations_ANN_fgsm.csv
Saved: ..\..\annotations\adv_annotations\Annotations_ANN_pgd.csv
Saved: ..\..\annotations\adv_annotations\Annotations_CNN_fgsm.csv
Saved: ..\..\annotations\adv_annotations\Annotations_CNN_pgd.csv
Saved: ..\..\annotations\adv_annotations\Annotations_RNN_fgsm.csv
Saved: ..\..\annotations\adv_annotations\Annotations_RNN_pgd.csv

Done. Relative-path annotations are in: ..\..\annotations\adv_annotations
